In [ ]:
Задание 1.1

In [1]:
import numpy as np

In [6]:
def chessboard(n):
    ind = np.arange(n)
    return(ind[:,None] + ind[None,:]) % 2

In [7]:
chessboard(5)

array([[0, 1, 0, 1, 0],
       [1, 0, 1, 0, 1],
       [0, 1, 0, 1, 0],
       [1, 0, 1, 0, 1],
       [0, 1, 0, 1, 0]], dtype=int32)

In [8]:
print(chessboard(5))

[[0 1 0 1 0]
 [1 0 1 0 1]
 [0 1 0 1 0]
 [1 0 1 0 1]
 [0 1 0 1 0]]


In [ ]:
Задание 1.2

In [9]:
def dfact_prod(n, dtype = object):
    return np.prod(np.arange(n, 0, -2, dtype=dtype))

In [10]:
def dfact_logspace(n):
    arr = np.arange(n, 0, -2, dtype=np.float64)
    return np.exp(np.sum(np.log(arr)))

In [11]:
def dfact_reduce(n, dtype=object):
    return np.multiply.reduce(np.arange(n, 0, -2, dtype=dtype))

In [13]:
print(dfact_prod(10))
print(dfact_logspace(10))
print(dfact_reduce(10))

3840
3839.9999999999977
3840


In [ ]:
Задание 1.3

In [28]:
import numpy as np


def _row_run_clues(grid):
    R, C = grid.shape
    W = C + 2                       

    flat = np.pad(grid, ((0, 0), (1, 1))).ravel()

    d = np.diff(np.concatenate(([0], flat, [0])))

    starts  = np.flatnonzero(d ==  1)
    ends    = np.flatnonzero(d == -1)
    lengths = ends - starts
    row_ids = starts // W

    counts = np.bincount(row_ids, minlength=R)

    order         = np.argsort(row_ids, kind="stable")
    group_offsets = np.concatenate(([0], np.cumsum(counts)))[:-1]
    pos_sorted    = np.arange(row_ids.size) - np.repeat(group_offsets, counts)
    pos           = np.empty(row_ids.size, dtype=int)
    pos[order]    = pos_sorted

    K = max(1, int(counts.max()))
    clues = np.zeros((R, K), dtype=int)
    clues[row_ids, pos] = lengths
    return clues, counts


def generate_nonogram(rows, cols, density=0.5, seed=None):
    rng  = np.random.default_rng(seed)
    grid = (rng.random((rows, cols)) < density).astype(np.uint8)

    row_clues, row_counts = _row_run_clues(grid)
    col_clues, col_counts = _row_run_clues(grid.T)

    return {
        "grid":       grid,
        "row_clues":  row_clues,
        "row_counts": row_counts,
        "col_clues":  col_clues,
        "col_counts": col_counts,
    }

In [29]:
def print_nonogram(puzzle):
    grid = puzzle["grid"]
    R, C = grid.shape
    rc, rn = puzzle["row_clues"], puzzle["row_counts"]
    cc, cn = puzzle["col_clues"], puzzle["col_counts"]
    MR, MC = rc.shape[1], cc.shape[1]

    # Верхние (столбцовые) подсказки
    for k in range(MC):
        parts = []
        for j in range(C):
            if k >= MC - cn[j]:
                parts.append(f"{cc[j, k - (MC - cn[j])]:>3}")
            else:
                parts.append("   ")
        print(" " * (3 * MR) + "  " + "".join(parts))
    print(" " * (3 * MR) + "  " + "───" * C)
    
    for i in range(R):
        left = []
        for k in range(MR):
            if k >= MR - rn[i]:
                left.append(f"{rc[i, k - (MR - rn[i])]:>3}")
            else:
                left.append("   ")
        cells = [" ■ " if v else " · " for v in grid[i]]
        print("".join(left) + "  " + "".join(cells))

In [41]:
seed=np.random.randint(0, 2**31)
puzzle = generate_nonogram(6, 8, density=0.55, seed=seed)
print_nonogram(puzzle)

                   1               
             1  2  1     1  2  2   
             1  1  2  2  1  2  1  1
           ────────────────────────
     2  2   ·  ■  ■  ·  ·  ■  ■  · 
  1  1  3   ·  ■  ·  ■  ·  ■  ■  ■ 
        3   ·  ·  ■  ■  ■  ·  ·  · 
        2   ■  ■  ·  ·  ·  ·  ·  · 
     1  2   ·  ·  ■  ·  ■  ■  ·  · 
  1  1  2   ■  ·  ■  ·  ·  ■  ■  · 
